In [1]:
from ipywidgets import HBox, VBox, Output, widgets, Image

from IPython.display import display_svg, display

In [96]:
from pygraphviz import AGraph
import re

class IGraph(AGraph):
    def __init__(self, thing=None, filename=None, data=None,
        string=None, handle=None, name='', strict=True,
        directed=False, styles=None, **attr,):
        self.styles = {} if styles is None else styles
        
        if string:
            for s in self.styles:            
                string = re.sub(fr'(\[[^\]]*?){s}\b', fr'\1{styles[s].to_str()}', string)
#         print(string)
        super().__init__(thing=thing, filename=filename, data=data, string=string,
                        handle=handle, name=name, strict=strict, directed=directed, **attr)
                
    def copy(self):
        from copy import deepcopy
        res = super().copy()
        res.styles = deepcopy(self.styles)
        return res
        
    def _repr_svg_(self):
        return str(self.draw(prog='dot', format='svg'))
    
    def select_edges(self, chain: str):
        for p in '-,>':
            chain = chain.replace(p, ' ')
        chain = chain.split()
        return [self.get_edge(*e) for e in zip(chain[:-1], chain[1:])]

    def style_edges(self, edges, attrs):
        if isinstance(edges, str):
            edges = self.select_edges(edges)
        if isinstance(attrs, str):
            attrs = self.styles[attrs]
        for e in edges:
            e.attr.update(attrs)
            
    def delete_node_edges(self, node, in_edges=True, out_edges=True):
        """ Delete the node with its edges."""
        if in_edges: self.delete_edges_from(self.in_edges(node))
        if out_edges: self.delete_edges_from(self.out_edges(node))
        self.delete_node(node)
            
class DotStyle(dict):
    def __init__(self, val=None, **kw):
        val = self.from_str(val) if isinstance(val, str) else {} if val is None else val
        assert isinstance(val, dict)
        val.update(kw)
        super().__init__(val)
        
    @staticmethod
    def from_str(val:str):
        import re

        def parse_val(s, sv):
            if sv:
                for op in (int, float, str):
                    try: s = op(sv)
                    except: continue  # if failed - continue for the next casting attempt
                    break  # otherwise - exit with success
            return s   # last cast option -> str always succeeds 

        found = re.findall(r'(\w+)\s*=\s*(?:"([^"]+)"|([^\s]+))', val)
        return {key: parse_val(s, sv) for key, s, sv in found}
    
    def to_str(self):
        return ' '.join([key + '=' + (f'"{v}"' if isinstance(v, str) and ' ' in v 
                             else f'{v}') for key, v in self.items()])
    
    def __iadd__(self, other):
        self.update(other)
        return self
    
    def __add__(self, other):
        res = self.__class__(self)
        res.update(other)
        return res
    
    def __radd__(self, other):
        return self.__class__(other) + self
    

In [70]:
# Prepare styles
def styles():
    resource = DotStyle(shape='rectangle', width=.9, height=.3, fixedsize='true',
                        fontname="Verdana", fontsize=10, penwidth=.5,
                        color=".14 .2 .4", style='filled', fillcolor=".14 0.2 1")
    produce = DotStyle(shape='oval', penwidth=0.5, fontsize=12)
    
    norm_edge = DotStyle(arrowsize=0.7, penwidth=0.5, color='gray')
    bold_edge = DotStyle(arrowsize=0.6, penwidth=1.5, color='black')
    war_edge = bold_edge + {'color': "0 0.5 1"}
    lord_edge = bold_edge + {'color': 'blue'}
    
    return locals()


In [107]:
dotsrc = """
digraph "Feaudal Processes" { 
    rankdir=LR
    ranksep=0.4
    newrank=true
    size = "5.5!"
    
    graph [label="Complete" labelloc=t]
    
    edge[norm_edge]
    node[produce] 

    {node[resource] 
        land peasants goods health protection labor
    } 
    
    subgraph cluster_c1 { label = "" color=white
        Reproduce -> peasants;   
        peasants -> Labor;
        {rank = min peasants Labor}
        {rank=same Reproduce }
    }
    
    goods -> Labor -> {land, labor} -> Work -> goods;
    
    goods -> Lord -> {protection, land};
    
    {goods, labor} -> Health -> health;
    {peasants, goods, labor, health} -> Reproduce;
    
    protection -> War -> {health, goods, land, peasants};  
    
        
    {rank=min   peasants Labor Lord  }
    {rank=same  land protection labor Reproduce}
    {rank=same  War Work Health  }
}

"""

G = IGraph(string=dotsrc, styles=styles())
G

In [5]:
!which dot

/opt/anaconda/envs/py36t/bin/dot


In [45]:
gs = [G]
gs.append(G.copy()); g = gs[-1]

g.graph_attr['label'] = 'Basic Scenario' 
lord_goods_chain = g.select_edges('Lord land Work goods Lord')
g.style_edges(lord_goods_chain, g.styles['bold_edge'] + {'color': 'blue'} )

peasant_goods_chain = g.select_edges('peasants Labor labor Work goods Reproduce peasants Reproduce')
g.style_edges(peasant_goods_chain, g.styles['bold_edge'] + {'color': 'black'})


other_edges = set(g.edges()).difference(peasant_goods_chain).difference(lord_goods_chain)
g.style_edges(other_edges, {'color': 'lightgray'})

g

KeyError: 'Node Lord not in graph.'

In [3]:
gs.append(gs[-1].copy()); g = gs[-1]

g.graph_attr['label'] = 'Basic War'

g.style_edges(g.select_edges('Lord protection War'), 'lord_edge')
g.style_edges(g.out_edges('War'), 'war_edge')


NameError: name 'gs' is not defined

In [4]:
outs = [Output() for _ in gs]
for o, g in zip(outs, gs):
    with o:
        display(g)

HBox(outs)

NameError: name 'gs' is not defined

In [9]:
class Lord(abe.Agent):
    def init(land, protection, households, greed=0):
        self.land = self.create('land', land)
        self.protection = self.create('protection', protection)
        self.households = households
        
    

NameError: name 'abe' is not defined

In [108]:
net_styles = dict(
    conv = DotStyle(shape='invtrapezium', fillcolor='lightblue', style='filled'),    
    relu = DotStyle(shape='Msquare', fillcolor='darkolivegreen2', style='filled'),
    conv_relu = DotStyle(color='darkolivegreen2'),
    pool = DotStyle(shape='invtriangle', fillcolor='orange', style='filled'),
    norm = DotStyle(shape='doublecircle', fillcolor='grey', style='filled'),
    full = DotStyle(shape='circle', fillcolor='salmon', style='filled'),
    drop = DotStyle(shape='tripleoctagon', fillcolor='plum2', style='filled'),
    
    to_conv = DotStyle(color='lightblue', style='bold'),
    to_norm = DotStyle(color='grey', style='bold'),
    to_pool = DotStyle(color='orange', style='bold'),
    
)


dotsrc_net = """
digraph Alexnet {    
    //  GRAPH OPTIONS                         
    rankdir=TB;         // From Top to Bottom
    size = "15!"

    labelloc="t";       // Tittle possition: top
    label="Alexnet";

    // =======================  NODES  ====================================
    data [shape=box3d, color=black];

    label [shape=tab, color=black];

    loss [shape=component, color=black];

    node [conv];
    conv1;
    conv3;
    
    node [relu];     // Rectified Linear Unit nodes
    relu1;
    relu3;
    relu6;
    relu7;
    
    
    // Splitted layer 2
    // ================
    //
    //  Layers with separated convolutions need to be in subgraphs
    //  This is because we want arrows from individual nodes but
    //  we want to consider all of them as a unique layer.
    //

    subgraph layer2 {        
        node [conv];     // Convolution nodes
        conv2_1;
        conv2_2;
        node [relu];     // Rectified Linear Unit nodes
        relu2_1;
        relu2_2;
    }

    // Splitted layer 4
    // ================    
    
    subgraph layer4 {   
        node [conv];    // Convolution nodes          
        conv3
        relu3
        conv4_1;
        conv4_2;
        node [relu];    // Rectified Linear Unit nodes
        relu4_1;
        relu4_2;
    }

    // Splitted layer 5
    // ================
    
    subgraph layer5{
        node [conv];    // Convolution nodes
        conv5_1;
        conv5_2;
        
        node [relu];
        relu5_1;
        relu5_2;
    }
        
    
    node [pool];     // Pooling nodes
    pool1;
    pool2;
    pool5;

    node [norm];  // Normalization nodes
    norm1;
    norm2;
            
    node [full];
    fc6;
    fc7;
    fc8;
            
    node [drop];
    drop6;
    drop7;

    // ===========================================  LAYESRS =================================
    
    // LAYER 1
    // -------------------------

    data -> conv1 [to_conv, label="out = 96, kernel = 11, stride = 4"];

    edge [conv_relu];
    conv1 -> relu1;
    relu1 -> conv1;

    conv1 -> norm1 [to_norm, label="local_size = 5, alpha = 0.0001, beta = 0.75"];
    norm1 -> pool1 [to_pool, label="pool = MAX, kernel = 3, stride = 2"];
    
    pool1 -> conv2_1 [to_conv, label="out = 256, kernel = 5, pad = 2"];
    pool1 -> conv2_2 [to_conv];

    // LAYER 2
    // --------------------------
    edge [conv_relu];
    conv2_1 -> relu2_1;
    conv2_2 -> relu2_2;
    relu2_1 -> conv2_1;
    relu2_2 -> conv2_2;

    conv2_1 -> norm2 [to_norm, label="local_size = 5, alpha = 0.0001, beta = 0.75"];
    conv2_2 -> norm2 [to_norm];
    norm2 -> pool2 [to_pool, label="pool = MAX, kernel = 3, stride = 2"];
    pool2 -> conv3 [to_conv, label="out = 384, kernel = 3, pad = 1"];
    
    // LAYER 3
    // -------------------------
    conv3 -> relu3 [conv_relu];
    relu3 -> conv3 [conv_relu];
    conv3 -> conv4_1 [to_conv, label="out = 384, kernel = 3, pad = 1"];
    conv3 -> conv4_2 [to_conv];
    
    // LAYER 4
    // ------------------
    edge [conv_relu];
    conv4_1 -> relu4_1;
    relu4_1 -> conv4_1;
    conv4_2 -> relu4_2;
    relu4_2 -> conv4_2;

    conv4_1 -> conv5_1 [to_conv, label="out = 256, kernel = 3, pad = 1"];
    conv4_2 -> conv5_2 [to_conv];

    
    // LAYER 5
    // ----------------------
    edge [conv_relu];
    conv5_1 -> relu5_1;
    relu5_1 -> conv5_1;
    conv5_2 -> relu5_2;
    relu5_2 -> conv5_2;

    conv5_1 -> pool5 [to_pool, label="pool = MAX, kernel = 3, stride = 2"];
    conv5_2 -> pool5 [to_pool];

    pool5 -> fc6 [color=salmon, style=bold, label="out = 4096"];
    fc6 -> relu6 [conv_relu];
    relu6 -> fc6 [conv_relu];
    fc6 -> drop6 [color=plum2, style=bold, label="dropout_ratio = 0.5"];
    drop6 -> fc6 [color=plum2];
    
    // LAYER 6
    // -----------------------
    fc6 -> fc7 [color=salmon, style=bold, label="out = 4096"];
    
    // LAYER 7
    // ----------------------
    fc7 -> relu7 [conv_relu];
    relu7 -> fc7 [conv_relu];
    fc7 -> drop7 [color=plum2, style=bold, label="dropout_ratio = 0.5"];
    drop7 -> fc7 [color=plum2];
    fc7 -> fc8 [color=salmon, style=bold, label="out = 1000"];
    
    // LAYER 8
    // ---------------------
    edge [color=black]
    fc8 -> loss;
    label -> loss;
}
"""


G = IGraph(string=dotsrc_net, styles=net_styles)
G.draw('om.png', )

AttributeError: Graph has no layout information, see layout() or specify prog=neato|dot|twopi|circo|fdp|nop.